# ⚙️ Pipeline completo: limpar os seus próprios registos de EEG

No notebook do workshop (`01_remocao_artefactos.ipynb`) vimos **porquê** e **como** se limpa um registo de EEG, passo a passo. Este notebook é a **ferramenta**: faz o mesmo trabalho de forma automática, com os seus próprios ficheiros.

### O que este pipeline faz
1. 📂 **Carrega** um registo (`.edf`, `.bdf`, `.fif`, `.set`, `.vhdr`, ...)
2. 🔌 **Remove o ruído da rede elétrica** (50 Hz na Europa, 60 Hz nos EUA, deteção automática)
3. 🎚️ **Filtra** as oscilações lentas e o ruído rápido
4. ⚡ **Deteta os elétrodos avariados** e reconstrói-os a partir dos vizinhos
5. 🍸 **Corre a ICA** e usa um **classificador treinado** (ICLabel) para reconhecer os componentes de pestanejo, músculo, coração, etc.
6. ✨ **Remove esses componentes**
7. 📥 **Guarda** o registo limpo e um **relatório** (uma página HTML com todos os gráficos), para descarregar

### Como usar
Corra as caixas por ordem (ou **Runtime → Run all**). Na caixa **📤 Carregar os seus ficheiros** abre-se um botão para escolher os registos. Se não escolher nenhum, usamos dados de exemplo.

## 0. Preparação

In [ ]:
# Instalar as ferramentas no Colab (cerca de um minuto). Escondemos as mensagens do pip,
# porque o Colab mostra avisos inofensivos ("dependency conflicts") sobre pacotes que não usamos.
import subprocess, sys
print("A instalar as ferramentas...")
result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                         "mne", "mne-icalabel", "onnxruntime", "pyprep"],
                        capture_output=True, text=True)
print("Pronto ✅" if result.returncode == 0 else result.stderr)

In [ ]:
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mne
from mne.datasets import eegbci
from mne_icalabel import label_components
from pyprep.find_noisy_channels import NoisyChannels
from google.colab import files

mne.set_log_level("ERROR")
mne.viz.set_browser_backend("matplotlib")
plt.rcParams["figure.dpi"] = 100

RAW_DIR = Path("data/raw")            # os registos originais
OUT_DIR = Path("data/preprocessed")   # os resultados
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 📤 Carregar os seus ficheiros

Ao correr esta caixa aparece um botão **"Choose Files"**. Escolha um ou mais registos.
- **BrainVision** (`.vhdr`): escolha também os ficheiros `.vmrk` e `.eeg` com o mesmo nome.
- **EEGLAB** (`.set`): se existir um ficheiro `.fdt` com o mesmo nome, escolha-o também.
- Para usar só os **dados de exemplo**, clique em **Cancel upload**.

In [ ]:
MAIN_EXTENSIONS = {".edf", ".bdf", ".fif", ".set", ".vhdr", ".cnt", ".gdf"}

uploaded = files.upload()
for fname in uploaded:
    Path(fname).rename(RAW_DIR / fname)   # mover para data/raw/

MY_FILES = [RAW_DIR / f for f in uploaded if Path(f).suffix.lower() in MAIN_EXTENSIONS]
print("Registos carregados:", [f.name for f in MY_FILES] or "nenhum (vamos usar os dados de exemplo)")

## ⚙️ Definições

Os valores por omissão funcionam bem para a maioria dos registos.

In [ ]:
# --- Elétrodos ---------------------------------------------------------------
MONTAGE = "standard_1005"   # posição dos elétrodos, se o ficheiro não a trouxer (sistema 10-20 / 10-05)

# --- Filtros -----------------------------------------------------------------
LINE_FREQ = None         # frequência da rede elétrica: 50, 60, ou None (= detetar automaticamente)
L_FREQ = 1.0             # remover tudo o que é mais lento do que isto (Hz)
H_FREQ = 40.0            # remover tudo o que é mais rápido do que isto (Hz), no registo final

# --- Elétrodos avariados -----------------------------------------------------
# Critérios do método PREP (biblioteca pyprep). Pode retirar critérios para ser menos exigente.
BAD_CHANNEL_CRITERIA = ["bad_by_nan", "bad_by_flat", "bad_by_deviation", "bad_by_hf_noise",
                        "bad_by_correlation", "bad_by_SNR", "bad_by_dropout", "bad_by_ransac"]

# --- ICA ---------------------------------------------------------------------
N_COMPONENTS = 20        # em quantos "componentes" separar o sinal
# Que tipos de componente remover, e com que certeza mínima do classificador (0 a 1)
REMOVE_LABELS = ["eye blink", "muscle artifact", "heart beat", "line noise", "channel noise"]
MIN_PROBABILITY = 0.80

## 🧰 As ferramentas do pipeline

Esta caixa define cada passo como uma função. **Não precisa de a alterar**: basta corrê-la.

In [ ]:
LABELS_PT = {"brain": "cérebro", "eye blink": "pestanejo / olhos", "muscle artifact": "músculo",
             "heart beat": "coração", "line noise": "rede elétrica", "channel noise": "elétrodo com ruído",
             "other": "outro"}


def load_recording(fname=None):
    # Ler um registo; sem ficheiro, descarregar um registo de exemplo (PhysioNet EEGBCI)
    if fname is None:
        fname = eegbci.load_data(1, [4], path=RAW_DIR, update_path=False)[0]
    raw = mne.io.read_raw(fname, preload=True)
    if raw.get_montage() is None:
        set_positions(raw)
    return raw, Path(fname).name.split(".")[0]


def set_positions(raw):
    # Cada sistema escreve os nomes à sua maneira ("FP1", "Fp1.", "EEG Fz"...):
    # tentamos fazê-los corresponder aos nomes da montagem padrão.
    standard = mne.channels.make_standard_montage(MONTAGE)
    lookup = {ch.lower(): ch for ch in standard.ch_names}
    rename = {}
    for ch in raw.ch_names:
        key = ch.strip(" .").lower().removeprefix("eeg ").strip()
        if key in lookup and lookup[key] not in raw.ch_names:
            rename[ch] = lookup[key]
    raw.rename_channels(rename)
    raw.set_montage(standard, on_missing="ignore")
    # Canais de EEG sem posição conhecida não podem ser usados: pô-los de parte
    unknown = [c["ch_name"] for c in raw.info["chs"]
               if c["kind"] == mne.io.constants.FIFF.FIFFV_EEG_CH and np.isnan(c["loc"][:3]).any()]
    if unknown:
        raw.set_channel_types({ch: "misc" for ch in unknown})
        print(f"⚠️ Sem posição conhecida, postos de parte: {', '.join(unknown)}")


def detect_line_freq(raw):
    # Procurar um pico estreito aos 50 ou aos 60 Hz (o "zumbido" da rede elétrica)
    nyquist = raw.info["sfreq"] / 2
    psd = raw.compute_psd(picks="eeg", fmin=40, fmax=min(70, nyquist - 1))
    power = 10 * np.log10(psd.get_data().mean(axis=0))
    best, best_peak = None, 3.0   # o pico tem de estar pelo menos 3 dB acima da vizinhança
    for f in (50, 60):
        if f < nyquist - 1:
            dist = np.abs(psd.freqs - f)
            peak = power[dist < 0.6].max() - np.median(power[(dist > 2) & (dist < 6)])
            if peak > best_peak:
                best, best_peak = f, peak
    return best


def filter_recording(raw, line_freq):
    # Notch na rede elétrica (e harmónicas) + filtro 1-100 Hz, como o ICLabel espera
    data = raw.copy()
    if line_freq:
        data.notch_filter(np.arange(line_freq, raw.info["sfreq"] / 2, line_freq), picks="eeg")
    return data.filter(l_freq=L_FREQ, h_freq=min(100.0, raw.info["sfreq"] / 2 - 5), picks="eeg")


def find_bad_channels(raw):
    # Método PREP: elétrodos planos, com demasiado ruído, ou que não "concordam" com os vizinhos
    nc = NoisyChannels(raw.copy().pick("eeg"), random_state=97)
    nc.find_all_bads(ransac="bad_by_ransac" in BAD_CHANNEL_CRITERIA)
    found = nc.get_bads(as_dict=True)
    return {ch: [c.removeprefix("bad_by_") for c in BAD_CHANNEL_CRITERIA if ch in found.get(c, [])]
            for ch in sorted({ch for c in BAD_CHANNEL_CRITERIA for ch in found.get(c, [])})}


def run_ica(raw):
    # O ICLabel foi treinado com ICA "extended infomax", referência média e filtro 1-100 Hz
    n_good = len(mne.pick_types(raw.info, eeg=True, exclude="bads"))
    ica = mne.preprocessing.ICA(n_components=min(N_COMPONENTS, n_good - 1), method="infomax",
                                fit_params=dict(extended=True), random_state=97, max_iter="auto")
    ica.fit(raw, picks="eeg")
    result = label_components(raw, ica, method="iclabel")
    labels, probs = result["labels"], result["y_pred_proba"]
    remove = [i for i, (l, p) in enumerate(zip(labels, probs)) if l in REMOVE_LABELS and p >= MIN_PROBABILITY]
    table = pd.DataFrame({"componente": [f"ICA{i:03d}" for i in range(len(labels))],
                          "tipo": [LABELS_PT[l] for l in labels],
                          "certeza": np.round(probs, 2),
                          "remover?": ["✂️ sim" if i in remove else "" for i in range(len(labels))]})
    return ica, remove, table


def save_results(name, raw, before_ica, cleaned, ica, removed, table, bads, line_freq):
    # Guardar o registo limpo, a ICA e um relatório HTML; devolve a lista de ficheiros
    paths = [OUT_DIR / f"{name}_clean-raw.fif", OUT_DIR / f"{name}-ica.fif", OUT_DIR / f"{name}_relatorio.html"]
    cleaned.save(paths[0], overwrite=True)
    ica.save(paths[1], overwrite=True)
    report = mne.Report(title=f"Limpeza do EEG: {name}")
    report.add_html(f"<p><b>Rede elétrica:</b> {line_freq or 'não detetada'} Hz<br>"
                    f"<b>Elétrodos reconstruídos:</b> {', '.join(bads) or 'nenhum'}<br>"
                    f"<b>Componentes removidos:</b> {', '.join(f'ICA{i:03d}' for i in removed) or 'nenhum'}</p>"
                    + table.to_html(index=False), title="Resumo")
    report.add_raw(raw, title="Registo original", psd=True, butterfly=False)
    report.add_ica(ica, title="ICA", inst=before_ica, n_jobs=1)
    report.add_raw(cleaned, title="Registo limpo", psd=True, butterfly=False)
    report.save(paths[2], overwrite=True, open_browser=False)
    return paths


def clean_recording(fname):
    # O pipeline inteiro, de uma só vez (usado na secção "todos os ficheiros")
    raw, name = load_recording(fname)
    line_freq = LINE_FREQ or detect_line_freq(raw)
    data = filter_recording(raw, line_freq)
    bads = list(find_bad_channels(data))
    data.info["bads"] = bads
    if bads:
        data.interpolate_bads()
    data.set_eeg_reference("average")
    ica, remove, table = run_ica(data)
    ica.exclude = remove
    cleaned = ica.apply(data.copy()).filter(l_freq=None, h_freq=H_FREQ, picks="eeg")
    paths = save_results(name, raw, data, cleaned, ica, remove, table, bads, line_freq)
    return dict(name=name, line_freq=line_freq, bads=bads, removed=remove, paths=paths)


def show(recordings, titles, channels=None, start=30, duration=10):
    # Mostrar o mesmo intervalo de tempo de vários registos, um por baixo do outro
    if channels is None:   # até 12 elétrodos espalhados pela cabeça, da testa para a nuca
        eeg = recordings[0].copy().pick("eeg")
        by_position = [c["ch_name"] for c in sorted(eeg.info["chs"], key=lambda c: -c["loc"][1])]
        channels = [by_position[i] for i in np.linspace(0, len(by_position) - 1, 12).astype(int)]
    start = min(start, recordings[0].times[-1] - duration)
    fig, axes = plt.subplots(len(recordings), 1, sharex=True, squeeze=False,
                             figsize=(12, 0.4 * len(channels) * len(recordings) + 1.5))
    spacing = None
    for ax, rec, title in zip(axes[:, 0], recordings, titles):
        data, times = rec.get_data(picks=channels, tmin=start, tmax=start + duration, return_times=True)
        data = (data - data.mean(axis=1, keepdims=True)) * 1e6
        spacing = spacing or 4 * np.median(np.std(data, axis=1))
        for i, trace in enumerate(data):
            ax.plot(times, trace - i * spacing, color="black", lw=0.6)
        ax.set_yticks(-np.arange(len(channels)) * spacing, channels)
        ax.set_ylim(-len(channels) * spacing, 1.5 * spacing)
        ax.set_title(title, loc="left", fontweight="bold")
        ax.spines[["top", "right", "left"]].set_visible(False)
    axes[-1, 0].set_xlabel("Tempo (segundos)")
    axes[-1, 0].set_xlim(times[0], times[-1])
    plt.tight_layout()
    plt.show()


def spectrum(recordings, labels, title):
    # Comparar o espetro (quanto existe de cada frequência) de vários registos
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for rec, label in zip(recordings, labels):
        psd = rec.compute_psd(picks="eeg")
        ax.plot(psd.freqs, 10 * np.log10(psd.get_data().mean(axis=0) * 1e12), label=label)
    ax.set_xlabel("Frequência (Hz)")
    ax.set_ylabel("Potência (dB)")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()


def download(paths, zip_name):
    # Juntar os ficheiros num .zip e descarregá-lo para o seu computador
    zip_path = OUT_DIR / zip_name
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            z.write(p, arcname=p.name)
    files.download(str(zip_path))

print("Ferramentas prontas ✅")

---
# Passo a passo (um registo)

Vamos limpar **um** registo e ver o que acontece em cada passo: o primeiro que carregou, ou os dados de exemplo.

## 1. 📂 Carregar o registo

In [ ]:
raw, name = load_recording(MY_FILES[0] if MY_FILES else None)

n_eeg = len(mne.pick_types(raw.info, eeg=True))
print(f"Registo              : {name}")
print(f"Elétrodos de EEG     : {n_eeg}  (outros canais: {len(raw.ch_names) - n_eeg})")
print(f"Duração              : {raw.times[-1]:.0f} segundos")
print(f"Medições por segundo : {raw.info['sfreq']:.0f}")

raw.plot_sensors(show_names=True, sphere="eeglab");

⚠️ **Confirme:** os elétrodos aparecem nos sítios certos da cabeça? Se o seu sistema tiver canais que *não* são EEG (olhos `EOG`, coração `ECG`, ...) mas aparecem como EEG, indique o tipo certo nesta caixa e volte a corrê-la, por exemplo: `raw.set_channel_types({"VEOG": "eog", "ECG": "ecg"})`.

## 2. 🔌 Rede elétrica e filtros

In [ ]:
line_freq = LINE_FREQ or detect_line_freq(raw)
print("Rede elétrica:", f"{line_freq} Hz" if line_freq else "não detetada")

filtered = filter_recording(raw, line_freq)
spectrum([raw, filtered], ["original", "filtrado"], "Espetro antes e depois dos filtros")

Nesta fase o filtro vai até aos 100 Hz (ou o máximo que o registo permite), porque o classificador do passo 4 foi treinado assim. O corte final nos **40 Hz** (`H_FREQ`) é feito no fim.

## 3. ⚡ Elétrodos avariados

In [ ]:
bads = find_bad_channels(filtered)
for ch, reasons in bads.items():
    print(f"  {ch:5s}  motivo: {', '.join(reasons)}")
print(f"{len(bads)} elétrodo(s) avariado(s)" if bads else "Nenhum elétrodo avariado 🎉")

Os motivos: **flat** = sinal plano; **deviation** = amplitude muito diferente dos outros; **hf_noise** = demasiado ruído rápido (muitas vezes músculo); **correlation** / **ransac** = não "concorda" com os vizinhos; **SNR** = mau em vários critérios; **dropout** = perde o sinal de vez em quando.

Agora reconstruímo-los a partir dos vizinhos:

In [ ]:
repaired = filtered.copy()
repaired.info["bads"] = list(bads)
if bads:
    repaired.interpolate_bads()
    show([filtered, repaired], ["Elétrodos avariados (originais)", "Depois de reconstruídos"],
         channels=list(bads)[:12])
repaired.set_eeg_reference("average");   # cada elétrodo passa a ser medido em relação à média de todos

## 4. 🍸 ICA e classificação automática dos componentes

A ICA separa o sinal em "vozes" (componentes). Depois, o **[ICLabel](https://mne.tools/mne-icalabel/)**, um classificador treinado com milhares de componentes etiquetados por especialistas, diz-nos o que é cada um.

In [ ]:
ica, remove, table = run_ica(repaired)
table

In [ ]:
ica.plot_components(sphere="eeglab");
print("Componentes a remover:", [f"ICA{i:03d}" for i in remove] or "nenhum")

👉 **Confira a decisão do classificador.** Se discordar, escolha à mão, por exemplo `remove = [0, 4]`, antes de continuar. Para ver um componente em detalhe: `ica.plot_properties(repaired, picks=[0])`.

## 5. ✨ Remover os componentes e comparar

In [ ]:
ica.exclude = remove
cleaned = ica.apply(repaired.copy()).filter(l_freq=None, h_freq=H_FREQ, picks="eeg")

only_filtered = raw.copy().filter(L_FREQ, H_FREQ, picks="eeg").set_eeg_reference("average")
show([only_filtered, cleaned], ["ANTES (só filtrado)", "DEPOIS (pipeline completo)"])
spectrum([raw, cleaned], ["original", "limpo"], "Espetro: original vs. limpo")

O espetro do registo limpo fica um pouco mais baixo: é o efeito de retirar os artefactos (que são grandes) e de usar a referência média. A queda depois dos 40 Hz é o filtro final.

## 6. 📥 Guardar e descarregar

In [ ]:
paths = save_results(name, raw, repaired, cleaned, ica, remove, table, list(bads), line_freq)
download(paths, f"{name}_resultados.zip")

O `.zip` tem:
- **`..._clean-raw.fif`**: o registo limpo, pronto para análise (abre-se com `mne.io.read_raw("...")`).
- **`..._relatorio.html`**: abra-o no navegador para ver todos os gráficos e decisões.
- **`...-ica.fif`**: a ICA calculada, para não ter de a repetir.

---
# 🚀 Limpar todos os ficheiros de uma vez

Quando estiver satisfeito com as definições, esta caixa aplica o pipeline a **todos os registos que carregou**, e descarrega os resultados num só `.zip`.

In [ ]:
summary, all_paths = [], []
for f in MY_FILES:
    print(f"A processar {f.name}...")
    res = clean_recording(f)
    all_paths += res["paths"]
    summary.append({"ficheiro": f.name, "rede elétrica (Hz)": res["line_freq"],
                    "elétrodos reconstruídos": ", ".join(res["bads"]) or "-",
                    "componentes removidos": ", ".join(f"ICA{i:03d}" for i in res["removed"]) or "-"})

if MY_FILES:
    download(all_paths, "todos_os_resultados.zip")
    display(pd.DataFrame(summary))
else:
    print("Não carregou nenhum registo na secção 📤.")